# Ch.1 — GPU Architecture Fundamentals (Azure Supplement)

**Azure-specific extension:** This notebook adapts the main `notebook.ipynb` concepts to Azure infrastructure. Instead of generic GPU specs, we explore:

- **Azure GPU VM SKUs** (NC-series, ND-series, NV-series, NCads-series)
- **Azure Pricing API** integration for real-time cost estimation
- **Azure ML compute** configuration for LLM workloads

**Prerequisites:**
- Complete the main `notebook.ipynb` first (understands roofline, arithmetic intensity, batch sizing)
- Azure subscription (Free Tier sufficient — all cells run locally, no GPU provisioning required)
- Azure CLI installed: `pip install azure-identity azure-mgmt-compute azure-mgmt-pricing`

**What you'll build:**
1. Azure GPU VM spec database — compare NC vs ND vs NV families
2. Real-time cost calculator using Azure Pricing API
3. Azure-specific roofline comparison (A100 vs V100 on Azure)
4. InferenceBase decision tool for Azure deployments (region + SKU selection)

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 1 · Azure GPU VM Spec Database

Azure offers several GPU VM families for AI/ML workloads:

| Family | GPU | Use Case | Pricing Tier |
|--------|-----|----------|-------------|
| **NC-series (v3)** | V100 (16GB) | Training, legacy inference | Mid |
| **NCas_T4_v3** | T4 (16GB) | Cost-optimized inference | Low |
| **ND-series (A100 v4)** | A100 (40/80GB) | Large-scale training, HPC | High |
| **ND_H100_v5** | H100 (80GB) | Frontier LLM training | Premium |
| **NV-series** | M60/T4 | Visualization, light AI | Low-Mid |

**Key differences from the main notebook:**
- Azure VMs include CPU, RAM, storage — not just GPU
- Pricing is per-hour, not one-time hardware cost
- Regional availability varies (H100 only in select regions as of 2026)
- NVLink availability depends on VM size (e.g., ND96 has 8×A100 with NVLink)

Below we build the same spec database, but for Azure GPU VMs.

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 2 · Azure Roofline Model Comparison

We apply the same roofline analysis from the main notebook, but now comparing Azure-available GPUs:

- **A100 80GB** (ND96/NC24ads): 312 TFLOPS, 2.0 TB/s, ridge = 156 FLOP/byte
- **V100 16/32GB** (NC6s/NC24s): 125 TFLOPS, 0.9 TB/s, ridge = 139 FLOP/byte
- **T4** (NC4as_T4): 65 TFLOPS, 0.32 TB/s, ridge = 203 FLOP/byte

The T4's higher ridge point (203 vs 156) reflects its lower bandwidth — operations must have higher arithmetic intensity to become compute-bound. For LLM decode (intensity ~1–8 FLOP/byte), T4 remains memory-bound but still cost-effective.

In [ ]:
# TODO: Implement this cell


## 3 · Azure Pricing Calculator

**Live pricing data:** If you provided Azure credentials above, this cell fetches real-time pricing from the Azure Retail Prices API. Otherwise, it uses the hardcoded estimates from our database.

**Cost model for InferenceBase on Azure:**
- **Scenario:** Llama-3-8B BF16, 5,000 tokens/sec aggregate throughput
- **Constraint:** <$15,000/month budget
- **Variables:** VM SKU, number of instances, region

The calculator estimates:
1. Tokens/sec per VM (bandwidth-limited)
2. Number of VMs required
3. Total monthly cost (compute + egress)
4. Break-even utilization (% uptime to justify reserved instance)

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


## 4 · Azure-Specific Considerations

**Regional availability:**
- H100 VMs: Only in select regions (East US, West Europe as of April 2026)
- Quota limits: New Azure subscriptions default to 0 GPU quota — must request increase
- Spot instances: 60–90% discount, but can be evicted (not shown in this notebook)

**Networking costs:**
- Egress: ~$0.05–0.12/GB (region-dependent)
- For LLM APIs serving 1 billion tokens/month, egress = ~4 TB = $200–480/mo
- InferenceBase should add this to total cost (not included in VM pricing above)

**Azure ML integration:**
- Managed endpoints add ~$0.10/hr/instance for orchestration
- Auto-scaling can reduce cost by 40–60% for variable traffic
- For production, consider Azure Container Instances (ACI) or AKS for lower overhead

**Optimization strategies:**
1. **Reserved Instances:** 1-year = 30% off, 3-year = 60% off (if committed volume)
2. **Spot VMs:** Best for batch inference, not real-time serving
3. **Batching:** Continuous batching (vLLM, Text Generation Inference) can 5–10× throughput
4. **Quantization:** INT8 halves memory/bandwidth → fit on cheaper VMs or double throughput

In [ ]:
# TODO: Implement this cell


## Summary

**What you learned (Azure-specific):**

1. **Azure GPU families** — NC (training/legacy), NCas_T4 (budget inference), ND (premium A100/H100)
2. **Roofline on Azure** — T4 has higher ridge (203) but lower peak; A100 best for large batches
3. **Cost modeling** — For InferenceBase's 5k tok/s @ $15k/mo, NC4as_T4 is the most cost-effective
4. **Azure-specific costs** — Egress, managed endpoints, quota limits all add to TCO
5. **RI economics** — 1-year RI saves 30% if uptime >70%; 3-year saves 60% but risky for fast-evolving LLM space

**Next steps:**
- Ch.2 Supplement: Azure ML memory profiling (track VRAM usage per VM SKU)
- Ch.3 Supplement: Deploy quantized model to Azure ML endpoint (INT8 → cheaper VMs)
- Ch.5 Supplement: Azure Kubernetes Service (AKS) + vLLM for production inference

**Bridge to Ch.2:** Now that we understand Azure GPU options and costs, Ch.2 will teach memory budgeting — how to calculate if a model fits in a given VM's VRAM, and how to predict KV cache growth under load.